In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install node2vec tensorflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.7/26.7 MB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 92.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.6/38.6 MB 47.6 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: scipy
    Found existing installation: scipy 1.16.1
    Uninstalling scipy-1.16.1:
      Successfully uninstalled scipy-1.16.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.12.0.88 requi

In [ ]:
 !pip uninstall -y jax numpy
 !pip cache purge

Found existing installation: numpy 1.23.5
Uninstalling numpy-1.23.5:
  Successfully uninstalled numpy-1.23.5
Files removed: 6


In [ ]:
 !pip install numpy==1.23.5

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 58.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
orbax-checkpoint 0.11.21 requires jax>=0.5.0, which is not installed.
optax 0.2.5 requires jax>=0.4.27, which is not installed.
dopamine-rl 4.1.2 requires jax>=0.1.72, which is not installed.
flax 0.10.6 requires jax>=0.5.1, which is not installed.
chex 0.1.90 requires jax>=0.4.27, which is not installed.
node2vec 0.5.0 requires numpy<2.0.0,>=1.24.0, but you have numpy 1.23.5 which is incompatible.
imbalanced-learn 0.13.0 requires numpy<3,>=1.24.3, but you have numpy 1.23.5 which is incompatible.
treescope 0.1.10 requires numpy>=1.25.2, but you have numpy 1.23.5 which is incompatible.
scikit-image 0.25.2 requires numpy>=1.24, but you have numpy 1.23.5 which is incompatible.
pymc 5.25.1 requires numpy>=1.25.0, but you have numpy 1.23.5 w

In [ ]:
import pandas as pd
from tqdm import tqdm
import json
import os
import random
import math
import pickle
import numpy as np
import scipy.sparse as sp
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelBinarizer
from sklearn.metrics import f1_score, roc_auc_score, average_precision_score, confusion_matrix
from collections import deque

import networkx as nx
import warnings
import keras
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras import activations, initializers, constraints, regularizers
from tensorflow.keras.layers import Input, Layer, Lambda, Dropout, Reshape, Dense, Embedding, LeakyReLU, Maximum
from tensorflow.keras.callbacks import EarlyStopping

from tensorflow.keras import layers, optimizers, losses, metrics, Model
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure magic command is only used in an interactive environment
try:
    get_ipython().run_line_magic('matplotlib', 'inline')
except NameError:
    pass


In [ ]:
dataFrame = pd.read_csv("/content/drive/MyDrive/Journal/citation_sentiment_corpus.txt", sep = "	", header = None)
dataFrame.columns = ["Source_PaperID", "Target_PaperID", "Sentiment", "Citation_text"]
dataFrame.Sentiment = dataFrame.Sentiment.replace({"o": 1,"p": 2,"n": 0})

/tmp/ipython-input-2858447318.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  dataFrame.Sentiment = dataFrame.Sentiment.replace({"o": 1,"p": 2,"n": 0})


In [ ]:
dataFrame.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8736 entries, 0 to 8735
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Source_PaperID  8736 non-null   object
 1   Target_PaperID  8736 non-null   object
 2   Sentiment       8736 non-null   int64 
 3   Citation_text   8736 non-null   object
dtypes: int64(1), object(3)
memory usage: 273.1+ KB


In [ ]:
Source = dataFrame['Source_PaperID']
Target = dataFrame['Target_PaperID']
Sentiment = dataFrame['Sentiment']

In [ ]:
import math

def directed_preferential_attachment(graph, edges):
    """
    Preferential Attachment for directed graphs:
    PA(u, v) = out_degree(u) * in_degree(v)
    """
    scores = []
    for u, v in edges:
        score = graph.out_degree(u) * graph.in_degree(v)
        scores.append((u, v, score))
    return scores


def directed_resource_allocation_index(graph, edges):
    """
    Resource Allocation Index for directed graphs:
    Use common nodes that are successors of u and predecessors of v.
    RA(u, v) = sum over w ∈ Γ_out(u) ∩ Γ_in(v) of 1 / degree(w)
    """
    scores = []
    for u, v in edges:
        common_neighbors = set(graph.successors(u)).intersection(set(graph.predecessors(v)))
        score = sum(1 / graph.degree(w) for w in common_neighbors if graph.degree(w) > 0)
        scores.append((u, v, score))
    return scores


def directed_jaccard_coefficient(graph, edges):
    """
    Jaccard Coefficient for directed graphs:
    JC(u, v) = |Γ_out(u) ∩ Γ_in(v)| / |Γ_out(u) ∪ Γ_in(v)|
    """
    scores = []
    for u, v in edges:
        succ_u = set(graph.successors(u))     # out-neighbors of u
        pred_v = set(graph.predecessors(v))   # in-neighbors of v
        intersection = succ_u & pred_v
        union = succ_u | pred_v
        score = len(intersection) / len(union) if union else 0
        scores.append((u, v, score))
    return scores


def directed_adamic_adar_index(graph, edges):
    """
    Adamic-Adar Index for directed graphs:
    AA(u, v) = sum over w ∈ Γ_out(u) ∩ Γ_in(v) of 1 / log(degree(w))
    """
    scores = []
    for u, v in edges:
        succ_u = set(graph.successors(u))     # out-neighbors of u
        pred_v = set(graph.predecessors(v))   # in-neighbors of v
        common_neighbors = succ_u & pred_v
        score = sum(1 / math.log(graph.degree(w)) for w in common_neighbors if graph.degree(w) > 1)
        scores.append((u, v, score))
    return scores


In [ ]:
import networkx as nx
import pandas as pd


# Create a DataFrame with source, target, and optional weights
edges = pd.DataFrame({
    "source": Source,
    "target": Target,
    "weight": [1] * len(Source)  # or use real weights if available
})

# Build a directed graph from the DataFrame
G = nx.from_pandas_edgelist(
    edges,
    source='source',
    target='target',
    edge_attr=True,  # includes all other columns (like 'weight') as edge attributes
    create_using=nx.DiGraph()
)


In [ ]:
# Create a leaderboard for nodes based on their degree (number of connections)
leaderboard = {node: G.degree(node) for node in G.nodes()}

# Convert the leaderboard to a pandas Series and then to a DataFrame
s = pd.Series(leaderboard, name="Citations")
citation_counts = s.to_frame().sort_values("Citations", ascending=False)

# Display the count of unique citation values
citation_value_counts = citation_counts.value_counts()

# Display results
print("Citation Counts:")
print(citation_counts.head())  # Top-ranked nodes by citations
print("\nValue Counts of Citations:")
print(citation_value_counts)


Citation Counts:
          Citations
J93-2004        436
J93-2003        368
P02-1040        305
P03-1021        281
N03-1017        240

Value Counts of Citations:
Citations
1            1755
2             684
3             283
4             129
5              77
6              26
7              14
8              13
9              11
10              7
11              5
15              4
17              4
20              4
14              4
22              3
13              2
18              2
71              2
27              2
30              2
12              2
151             1
101             1
109             1
117             1
121             1
125             1
138             1
172             1
152             1
368             1
182             1
212             1
240             1
281             1
305             1
95              1
31              1
67              1
29              1
16              1
19              1
23              1
24              1
25             

In [ ]:
print("Number of Nodes: ",G.number_of_nodes())
print("Number of Edges: ",G.number_of_edges())

Number of Nodes:  3069
Number of Edges:  5042


In [ ]:
#!pip install node2vec

In [ ]:
df=dataFrame
import pandas as pd
import numpy as np
from node2vec import Node2Vec
import tensorflow_hub as hub
import tensorflow as tf


In [ ]:
leaderboard = {}
for x in G.nodes:
 leaderboard[x] = len(G[x])
s = pd.Series(leaderboard, name='Citations')
citation_counts = s.to_frame().sort_values('Citations', ascending=False)
citation_counts.value_counts()

,count
Citations,
1,1785
2,703
3,295
4,121
0,77
5,59
6,19
7,7
8,3


In [ ]:
citation_counts = citation_counts.reset_index(level=0)
citation_counts.columns = ['Node', 'Citations']
citation_counts.head()

,Node,Citations
0,W08-0306,8
1,N09-1058,8
2,D07-1070,8
3,N09-1049,7
4,P08-1068,7


In [ ]:
print("Number of Nodes: ",G.number_of_nodes())
print("Number of Edges: ",G.number_of_edges())

Number of Nodes:  3069
Number of Edges:  5042


In [ ]:
zero_list = []
for i,j in zip(citation_counts['Node'], citation_counts['Citations']):
    if(j == 0):

        zero_list.append(i)
G.remove_nodes_from(zero_list)

In [ ]:

# ----- NODE-LEVEL FEATURES -----

# In-degree, Out-degree, Total Degree Centrality
in_degrees = dict(G.in_degree())
out_degrees = dict(G.out_degree())
total_degrees = {node: in_degrees[node] + out_degrees[node] for node in G.nodes()}

# Betweenness, Closeness, Eigenvector Centrality
betweenness = nx.betweenness_centrality(G)
closeness = nx.closeness_centrality(G)
eigenvector_centrality = nx.eigenvector_centrality(G)

# Combine Node Features into a DataFrame
node_features = pd.DataFrame({
    "node": list(G.nodes()),
    "in_degree": [in_degrees[node] for node in G.nodes()],
    "out_degree": [out_degrees[node] for node in G.nodes()],
    "total_degree": [total_degrees[node] for node in G.nodes()],
    "betweenness": [betweenness[node] for node in G.nodes()],
    "closeness": [closeness[node] for node in G.nodes()],
    "eigenvector_centrality": [eigenvector_centrality[node] for node in G.nodes()],
})


# ----- EDGE-LEVEL FEATURES USING CUSTOM FUNCTIONS -----
# Preferential Attachment
pref_attach = directed_preferential_attachment(G, G.edges())

# Resource Allocation Index
resource_allocation_scores = directed_resource_allocation_index(G, G.edges())

# Jaccard Coefficient
jaccard_scores = directed_jaccard_coefficient(G, G.edges())

# Adamic-Adar Index
adamic_adar_scores = directed_adamic_adar_index(G, G.edges())

# Combine Edge Features into a DataFrame
edge_features = pd.DataFrame({
    "source": [u for u, v, _ in pref_attach],
    "target": [v for u, v, _ in pref_attach],
    "preferential_attachment": [score for _, _, score in pref_attach],
    "resource_allocation": [score for _, _, score in resource_allocation_scores],
    "jaccard": [score for _, _, score in jaccard_scores],
    "adamic_adar": [score for _, _, score in adamic_adar_scores],
})



In [ ]:
print(edge_features.describe())

       preferential_attachment  resource_allocation      jaccard  adamic_adar
count              1589.000000          1589.000000  1589.000000  1589.000000
mean                141.224670             0.011998     0.002853     0.053190
std                 177.346121             0.050893     0.015371     0.179574
min                   1.000000             0.000000     0.000000     0.000000
25%                  20.000000             0.000000     0.000000     0.000000
50%                  68.000000             0.000000     0.000000     0.000000
75%                 169.000000             0.000000     0.000000     0.000000
max                1185.000000             0.666667     0.250000     1.820478


In [ ]:
#further Improvement and Normalization

path_lengths = []
for u, v in G.edges():
    try:
        path_length = nx.shortest_path_length(G, u, v)
    except nx.NetworkXNoPath:
        path_length = -1  # or a high value
    path_lengths.append(path_length)

edge_features["shortest_path_length"] = path_lengths


In [ ]:
node_features.shape , edge_features.shape

((2992, 7), (1589, 7))

In [ ]:
node_features.columns, edge_features.columns

(Index(['node', 'in_degree', 'out_degree', 'total_degree', 'betweenness',
        'closeness', 'eigenvector_centrality'],
       dtype='object'),
 Index(['source', 'target', 'preferential_attachment', 'resource_allocation',
        'jaccard', 'adamic_adar', 'shortest_path_length'],
       dtype='object'))

In [ ]:
print("Number of Nodes: ",G.number_of_nodes())
print("Number of Edges: ",G.number_of_edges())

Number of Nodes:  2992
Number of Edges:  1589


In [ ]:
# ----- STEP 2: NODE2VEC EMBEDDINGS -----
node2vec = Node2Vec(G, dimensions=128, walk_length=80, num_walks=10, workers=4)
node2vec_model = node2vec.fit(window=10, min_count=1, batch_words=4)

# Create a DataFrame for Node2Vec embeddings
node2vec_embeddings = pd.DataFrame(
    [node2vec_model.wv[str(node)] for node in G.nodes()],
    index=[str(node) for node in G.nodes()],
    columns=[f"node2vec_dim_{i}" for i in range(128)]
)


Computing transition probabilities:   0%|          | 0/2992 [00:00<?, ?it/s]

In [ ]:
node2vec_embeddings.head()

,node2vec_dim_0,node2vec_dim_1,node2vec_dim_2,node2vec_dim_3,node2vec_dim_4,node2vec_dim_5,node2vec_dim_6,node2vec_dim_7,node2vec_dim_8,node2vec_dim_9,...,node2vec_dim_118,node2vec_dim_119,node2vec_dim_120,node2vec_dim_121,node2vec_dim_122,node2vec_dim_123,node2vec_dim_124,node2vec_dim_125,node2vec_dim_126,node2vec_dim_127
A00-1043,0.002768,-0.003683,0.006099,-0.003521,0.002698,-0.006015,-0.005422,-0.000623,0.001502,0.001719,...,-0.005015,0.001011,0.006838,-0.003003,0.003509,0.005229,-0.001227,-0.003954,0.004361,-0.005689
H05-1033,-0.000110,-0.002600,0.007279,0.005711,-0.004930,0.004489,0.002117,-0.002198,-0.002542,0.002088,...,0.003482,-0.001324,0.004283,-0.004105,0.005587,-0.003236,-0.003738,0.002869,0.001316,0.006168
I05-2009,-0.001829,-0.004785,-0.000125,0.006509,0.002468,-0.001951,0.004048,-0.001546,0.001436,0.007350,...,0.002126,0.005980,0.002459,0.000205,0.000269,-0.007550,0.001097,-0.001950,0.006369,-0.003777
I08-1016,0.003937,-0.005712,-0.004640,-0.000452,-0.007326,-0.000736,0.000260,0.001043,-0.001319,0.007257,...,0.005938,0.003479,-0.004789,0.007074,-0.000776,0.000489,-0.002335,-0.001288,0.002949,-0.006420
I08-2101,-0.002044,0.004219,-0.007270,-0.003808,0.004117,-0.007357,0.007434,0.002026,0.000972,-0.001688,...,0.000516,-0.003904,-0.002677,-0.001902,0.000973,0.001094,0.002977,0.001686,-0.006424,0.001769


In [ ]:
# Ensure Node2Vec embeddings cover all nodes in the original DataFrame
all_nodes = df["Source_PaperID"].unique()
missing_nodes = set(all_nodes) - set(node2vec_embeddings.index)
for node in missing_nodes:
    node2vec_embeddings.loc[node] = np.zeros(128)

In [ ]:
# Reindex to ensure the order matches `all_nodes`
node2vec_embeddings = node2vec_embeddings.reindex(all_nodes)

In [ ]:
node2vec_embeddings.shape

(2992, 128)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
import tensorflow_hub as hub
import tensorflow as tf

# Node2Vec embeddings with `Source_PaperID` as the index
node2vec_embeddings.index.name = "Source_PaperID"

In [ ]:

# ----- STEP 2: EXTRACT CITATION TEXTS FOR UNIQUE NODES -----
# Select only rows where `Source_PaperID` matches `node2vec_embeddings` index
citation_texts = df[df["Source_PaperID"].isin(node2vec_embeddings.index)]
citation_texts.shape

(8736, 4)

In [ ]:
# Ensure unique rows for `Source_PaperID` (if duplicates exist, group by and take the first)
unique_citation_texts = citation_texts.groupby("Source_PaperID").first().reset_index()

In [ ]:
unique_citation_texts.shape

(2992, 4)

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from tqdm import tqdm

# ----- STEP 1: LOAD BERT ON CUDA -----
bert_model_name = "bert-base-uncased"  # Change to any BERT variant you prefer
tokenizer = AutoTokenizer.from_pretrained(bert_model_name)
bert_model = AutoModel.from_pretrained(bert_model_name).to("cuda")  # Move model to GPU

# Function to compute BERT embeddings for texts
def compute_bert_embeddings(texts):
    embeddings = []
    for text in tqdm(texts, desc="Processing BERT embeddings"):
        # Tokenize and encode text
        inputs = tokenizer(
            text, return_tensors="pt", truncation=True, padding="max_length", max_length=512
        ).to("cuda")  # Move inputs to GPU

        # Forward pass through the BERT model
        with torch.no_grad():
            outputs = bert_model(**inputs)

        # Use the [CLS] token embedding as the sentence representation
        cls_embedding = outputs.last_hidden_state[:, 0, :].squeeze().cpu().numpy()
        embeddings.append(cls_embedding)
    return np.array(embeddings)

# Ensure unique rows for `Source_PaperID`
unique_citation_texts = citation_texts.groupby("Source_PaperID").first().reset_index()

# Extract citation texts and their corresponding `Source_PaperID`
texts_to_encode = unique_citation_texts["Citation_text"].tolist()
source_ids = unique_citation_texts["Source_PaperID"].tolist()

# ----- STEP 3: COMPUTE BERT EMBEDDINGS -----
bert_embeddings = compute_bert_embeddings(texts_to_encode)

# Reduce BERT embeddings to 128 dimensions using PCA
pca = PCA(n_components=128)
reduced_bert_embeddings = pca.fit_transform(bert_embeddings)

# Create a DataFrame for reduced BERT embeddings
bert_embeddings_df = pd.DataFrame(
    reduced_bert_embeddings,
    index=source_ids,  # Use `Source_PaperID` as the index
    columns=[f"bert_dim_{i}" for i in range(128)]
)

# ----- STEP 4: MERGE NODE2VEC AND BERT -----
# Concatenate Node2Vec and reduced BERT embeddings
combined_embeddings = bert_embeddings_df

# ----- STEP 5: ADD SENTIMENT LABELS -----
# Map sentiment labels from the original DataFrame
sentiment_dict = df.set_index("Source_PaperID")["Sentiment"].to_dict()
combined_embeddings["sentiment"] = combined_embeddings.index.map(sentiment_dict)

# ----- STEP 6: FINAL DATAFRAME -----
# Reset index for final output
final_df = combined_embeddings.reset_index()

# Final DataFrame includes:
# - `Source_PaperID`: Node ID.
# - `node2vec_dim_*`: Node2Vec embeddings (128 dimensions).
# - `bert_dim_*`: BERT embeddings (128 dimensions).
# - `sentiment`: Sentiment label (target value).
print(final_df.head())
print("\nShape of the final DataFrame:", final_df.shape)


Processing BERT embeddings: 100%|██████████| 2992/2992 [00:42<00:00, 70.41it/s]


      index  bert_dim_0  bert_dim_1  bert_dim_2  bert_dim_3  bert_dim_4  \
0  A00-1004   -1.416983    3.080562    0.883313    0.436437    0.312889   
1  A00-1005   -1.938541    2.263871    0.381171    0.042236   -1.363889   
2  A00-1007    0.091380   -2.042169    3.897649    1.959164   -0.314125   
3  A00-1011   -0.618929   -1.249520   -0.298860   -1.988914   -1.755679   
4  A00-1012    0.032313    0.348407    0.054108    1.618243   -0.744880   

   bert_dim_5  bert_dim_6  bert_dim_7  bert_dim_8  ...  bert_dim_119  \
0   -0.342492    0.280365    0.529431    0.012933  ...      0.031464   
1   -0.678148   -0.588819    1.265437    0.315043  ...      0.482160   
2    1.525761    0.285054   -0.331658    0.313625  ...     -0.335718   
3   -0.538418    0.358828   -1.728716    0.044992  ...      0.148105   
4    0.196882    0.720973   -0.390180    1.085731  ...     -0.587456   

   bert_dim_120  bert_dim_121  bert_dim_122  bert_dim_123  bert_dim_124  \
0      0.054942      0.082063     -0.1672

In [ ]:
final_df.head()

,index,bert_dim_0,bert_dim_1,bert_dim_2,bert_dim_3,bert_dim_4,bert_dim_5,bert_dim_6,bert_dim_7,bert_dim_8,...,bert_dim_119,bert_dim_120,bert_dim_121,bert_dim_122,bert_dim_123,bert_dim_124,bert_dim_125,bert_dim_126,bert_dim_127,sentiment
0,A00-1004,-1.416983,3.080562,0.883313,0.436437,0.312889,-0.342492,0.280365,0.529431,0.012933,...,0.031464,0.054942,0.082063,-0.167240,0.064560,0.178628,-0.233630,-0.168098,-0.320026,1
1,A00-1005,-1.938541,2.263871,0.381171,0.042236,-1.363889,-0.678148,-0.588819,1.265437,0.315043,...,0.482160,0.045421,-0.149316,-0.351725,-0.173369,-0.285735,-0.108284,0.209745,0.208899,2
2,A00-1007,0.091380,-2.042169,3.897649,1.959164,-0.314125,1.525761,0.285054,-0.331658,0.313625,...,-0.335718,-0.184003,0.330230,0.107377,0.425395,-0.092539,-0.592054,0.298351,0.221422,1
3,A00-1011,-0.618929,-1.249520,-0.298860,-1.988914,-1.755679,-0.538418,0.358828,-1.728716,0.044992,...,0.148105,0.126957,-0.190360,0.250209,0.373774,-0.072877,0.565949,0.238295,-0.130621,1
4,A00-1012,0.032313,0.348407,0.054108,1.618243,-0.744880,0.196882,0.720973,-0.390180,1.085731,...,-0.587456,-0.071521,-0.198800,-0.054102,0.084508,-0.431686,0.298340,-0.143359,-0.168911,1


In [ ]:
dataset = final_df

In [ ]:
dataset.rename(columns={"Source_PaperID": "Node_id"}, inplace=True)
dataset.rename(columns={"sentiment": "label"}, inplace=True)

In [ ]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2992 entries, 0 to 2991
Columns: 130 entries, index to label
dtypes: float32(128), int64(1), object(1)
memory usage: 1.5+ MB


In [ ]:
node2vec_cc = dataset

In [ ]:
graph = nx.to_pandas_edgelist(G)

graph

,source,target,weight
0,W09-0604,N03-1003,1
1,A00-1031,W96-0213,1
2,A97-1004,W96-0213,1
3,C08-1026,P08-1085,1
4,N01-1023,W96-0213,1
...,...,...,...
1584,W07-1209,W96-0213,1
1585,W07-1516,W96-0213,1
1586,W07-2053,W96-0213,1
1587,W08-0611,W96-0213,1


In [ ]:
node2vec_cc["Node_id"] = dataset["index"]
node2vec_cc["label"] = dataset["label"]

In [ ]:
node2vec_cc.drop(columns = ["index"], inplace = True)

In [ ]:
paper_idx = {name: idx for idx, name in enumerate(node2vec_cc["Node_id"])}

node2vec_cc["Node_id"] = node2vec_cc["Node_id"].apply(lambda name: paper_idx[name])
graph["source"] = graph["source"].apply(lambda name: paper_idx[name])
graph["target"] = graph["target"].apply(lambda name: paper_idx[name])

In [ ]:
node2vec_cc.columns

Index(['bert_dim_0', 'bert_dim_1', 'bert_dim_2', 'bert_dim_3', 'bert_dim_4',
       'bert_dim_5', 'bert_dim_6', 'bert_dim_7', 'bert_dim_8', 'bert_dim_9',
       ...
       'bert_dim_120', 'bert_dim_121', 'bert_dim_122', 'bert_dim_123',
       'bert_dim_124', 'bert_dim_125', 'bert_dim_126', 'bert_dim_127', 'label',
       'Node_id'],
      dtype='object', length=130)

In [ ]:
node2vec_cc.head()

,bert_dim_0,bert_dim_1,bert_dim_2,bert_dim_3,bert_dim_4,bert_dim_5,bert_dim_6,bert_dim_7,bert_dim_8,bert_dim_9,...,bert_dim_120,bert_dim_121,bert_dim_122,bert_dim_123,bert_dim_124,bert_dim_125,bert_dim_126,bert_dim_127,label,Node_id
0,-1.416983,3.080562,0.883313,0.436437,0.312889,-0.342492,0.280365,0.529431,0.012933,-0.416567,...,0.054942,0.082063,-0.167240,0.064560,0.178628,-0.233630,-0.168098,-0.320026,1,0
1,-1.938541,2.263871,0.381171,0.042236,-1.363889,-0.678148,-0.588819,1.265437,0.315043,0.491038,...,0.045421,-0.149316,-0.351725,-0.173369,-0.285735,-0.108284,0.209745,0.208899,2,1
2,0.091380,-2.042169,3.897649,1.959164,-0.314125,1.525761,0.285054,-0.331658,0.313625,0.977610,...,-0.184003,0.330230,0.107377,0.425395,-0.092539,-0.592054,0.298351,0.221422,1,2
3,-0.618929,-1.249520,-0.298860,-1.988914,-1.755679,-0.538418,0.358828,-1.728716,0.044992,0.667788,...,0.126957,-0.190360,0.250209,0.373774,-0.072877,0.565949,0.238295,-0.130621,1,3
4,0.032313,0.348407,0.054108,1.618243,-0.744880,0.196882,0.720973,-0.390180,1.085731,-0.694201,...,-0.071521,-0.198800,-0.054102,0.084508,-0.431686,0.298340,-0.143359,-0.168911,1,4


In [ ]:
df=node2vec_cc

In [ ]:
node2vec_cc

,bert_dim_0,bert_dim_1,bert_dim_2,bert_dim_3,bert_dim_4,bert_dim_5,bert_dim_6,bert_dim_7,bert_dim_8,bert_dim_9,...,bert_dim_120,bert_dim_121,bert_dim_122,bert_dim_123,bert_dim_124,bert_dim_125,bert_dim_126,bert_dim_127,label,Node_id
0,-1.416983,3.080562,0.883313,0.436437,0.312889,-0.342492,0.280365,0.529431,0.012933,-0.416567,...,0.054942,0.082063,-0.167240,0.064560,0.178628,-0.233630,-0.168098,-0.320026,1,0
1,-1.938541,2.263871,0.381171,0.042236,-1.363889,-0.678148,-0.588819,1.265437,0.315043,0.491038,...,0.045421,-0.149316,-0.351725,-0.173369,-0.285735,-0.108284,0.209745,0.208899,2,1
2,0.091380,-2.042169,3.897649,1.959164,-0.314125,1.525761,0.285054,-0.331658,0.313625,0.977610,...,-0.184003,0.330230,0.107377,0.425395,-0.092539,-0.592054,0.298351,0.221422,1,2
3,-0.618929,-1.249520,-0.298860,-1.988914,-1.755679,-0.538418,0.358828,-1.728716,0.044992,0.667788,...,0.126957,-0.190360,0.250209,0.373774,-0.072877,0.565949,0.238295,-0.130621,1,3
4,0.032313,0.348407,0.054108,1.618243,-0.744880,0.196882,0.720973,-0.390180,1.085731,-0.694201,...,-0.071521,-0.198800,-0.054102,0.084508,-0.431686,0.298340,-0.143359,-0.168911,1,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2987,-0.368425,-1.685204,-1.407730,0.480121,1.511461,-1.247820,-0.357996,-0.251869,-1.078697,0.278451,...,-0.187505,-0.090190,0.004580,0.196960,-0.024187,0.087868,0.279540,0.131400,1,2987
2988,0.054088,-0.827501,-0.804189,-0.229911,-1.156996,-0.827191,-0.764786,-0.025073,1.580656,1.691621,...,-0.008688,0.102943,0.230320,0.065414,-0.208264,-0.002526,0.012509,0.026181,1,2988
2989,-2.638431,2.506631,0.022421,-0.446569,2.085835,0.664643,-0.443206,0.383283,-0.671917,-0.035076,...,-0.123553,-0.055213,-0.197013,-0.260603,0.113986,-0.176083,0.265351,-0.268781,2,2989
2990,-2.373599,-0.931010,-1.018901,0.500352,-0.241553,-0.645499,0.300079,-0.815610,-0.676226,-0.159109,...,-0.013784,0.129569,0.108512,0.034553,0.387114,0.061899,0.034430,-0.196584,1,2990


In [ ]:
df=node2vec_cc

In [ ]:
df = df.drop(columns=["Node_id"])

In [ ]:
# Features and labels
X = df.drop(columns=["label"]).values
y = df["label"].values

In [ ]:
# ---- First split: 70% train, 30% temp (val+test) ----
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)

# ---- Second split: from temp, split into 60% val and 40% test ----
# (because 0.60 * 0.30 = 0.18 and 0.40 * 0.30 = 0.12)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.40, stratify=y_temp, random_state=42
)

In [ ]:
import numpy as np

val_counts = np.unique(y_val, return_counts=True)
test_counts = np.unique(y_test, return_counts=True)
train_counts = np.unique(y_train, return_counts=True)

print("Validation set value counts:", dict(zip(val_counts[0], val_counts[1])))
print("Testing set value counts:", dict(zip(test_counts[0], test_counts[1])))
print("Training set value counts:", dict(zip(train_counts[0], train_counts[1])))

Validation set value counts: {0: 14, 1: 471, 2: 53}
Testing set value counts: {0: 9, 1: 316, 2: 35}
Training set value counts: {0: 53, 1: 1835, 2: 206}


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# create a logistic regression model
model = LogisticRegression()

# fit the model to the training data
model.fit(X_train, y_train)

# make predictions on the training, testing, and validation data
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)
y_val_pred = model.predict(X_val)

# evaluate the accuracy of the model on the training, testing, and validation data
Lr_test_accuracy = accuracy_score(y_test, y_test_pred)
Lr_val_accuracy = accuracy_score(y_val, y_val_pred)

print(f"Testing accuracy: {Lr_test_accuracy}")
print(f"Validation accuracy: {Lr_val_accuracy}")


Testing accuracy: 0.8694444444444445
Validation accuracy: 0.8643122676579925


#Scores

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# create a logistic regression model
model = LogisticRegression()

# fit the model to the training data
model.fit(X_train, y_train)

# make predictions
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)
y_val_pred = model.predict(X_val)

# evaluate accuracy
Lr_test_accuracy = accuracy_score(y_test, y_test_pred)
Lr_val_accuracy = accuracy_score(y_val, y_val_pred)

print(f"Testing accuracy: {Lr_test_accuracy:.4f}")
print(f"Validation accuracy: {Lr_val_accuracy:.4f}")

# macro, micro, and weighted metrics for test set
for avg in ['macro', 'micro', 'weighted']:
    print(f"\n--- {avg.upper()} ---")
    print(f"Precision: {precision_score(y_test, y_test_pred, average=avg):.4f}")
    print(f"Recall:    {recall_score(y_test, y_test_pred, average=avg):.4f}")
    print(f"F1-score:  {f1_score(y_test, y_test_pred, average=avg):.4f}")


Testing accuracy: 0.8694
Validation accuracy: 0.8643

--- MACRO ---
Precision: 0.4881
Recall:    0.3746
F1-score:  0.3787

--- MICRO ---
Precision: 0.8694
Recall:    0.8694
F1-score:  0.8694

--- WEIGHTED ---
Precision: 0.8120
Recall:    0.8694
F1-score:  0.8251


In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Define the parameter grid to search
param_grid = {
    'max_depth': [5, 10, 15, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# Initialize the Decision Tree classifier
dt_model = DecisionTreeClassifier()

# Use GridSearchCV to find the best hyperparameters
grid_search = GridSearchCV(estimator=dt_model, param_grid=param_grid, cv=5, scoring='accuracy', verbose=1, n_jobs=-1)

# Fit GridSearchCV to the training data
grid_search.fit(X_train, y_train)

# Get the best parameters and best estimator
best_params = grid_search.best_params_
best_model = grid_search.best_estimator_

print("Best parameters found: ", best_params)

# Evaluate on validation data
y_pred_val = best_model.predict(X_val)
Dt_val_accuracy = accuracy_score(y_val, y_pred_val)
print(f"\nValidation accuracy: {Dt_val_accuracy:.4f}")
for avg in ['macro', 'micro', 'weighted']:
    print(f"\nValidation ({avg.upper()}):")
    print(f"Precision: {precision_score(y_val, y_pred_val, average=avg):.4f}")
    print(f"Recall:    {recall_score(y_val, y_pred_val, average=avg):.4f}")
    print(f"F1-score:  {f1_score(y_val, y_pred_val, average=avg):.4f}")

# Evaluate on test data
y_pred_test = best_model.predict(X_test)
Dt_test_accuracy = accuracy_score(y_test, y_pred_test)
print(f"\nTesting accuracy: {Dt_test_accuracy:.4f}")
for avg in ['macro', 'micro', 'weighted']:
    print(f"\nTesting ({avg.upper()}):")
    print(f"Precision: {precision_score(y_test, y_pred_test, average=avg):.4f}")
    print(f"Recall:    {recall_score(y_test, y_pred_test, average=avg):.4f}")
    print(f"F1-score:  {f1_score(y_test, y_pred_test, average=avg):.4f}")



Fitting 5 folds for each of 36 candidates, totalling 180 fits
Best parameters found:  {'max_depth': 5, 'min_samples_leaf': 1, 'min_samples_split': 2}

Validation accuracy: 0.8625

Validation (MACRO):
Precision: 0.2913
Recall:    0.3284
F1-score:  0.3087

Validation (MICRO):
Precision: 0.8625
Recall:    0.8625
F1-score:  0.8625

Validation (WEIGHTED):
Precision: 0.7650
Recall:    0.8625
F1-score:  0.8108

Testing accuracy: 0.8694

Testing (MACRO):
Precision: 0.2923
Recall:    0.3302
F1-score:  0.3101

Testing (MICRO):
Precision: 0.8694
Recall:    0.8694
F1-score:  0.8694

Testing (WEIGHTED):
Precision: 0.7696
Recall:    0.8694
F1-score:  0.8165


In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Define the parameter grid to search
param_grid = {
    'C': [0.1, 1, 10, 100],
    'kernel': ['linear', 'rbf', 'poly', 'sigmoid'],
    'gamma': ['scale', 'auto']
}

# Initialize the SVM classifier
svm_model = SVC()

# Use GridSearchCV to find the best hyperparameters
grid_search = GridSearchCV(estimator=svm_model, param_grid=param_grid, cv=5,
                           scoring='accuracy', verbose=1, n_jobs=-1)

# Fit GridSearchCV to the training data
grid_search.fit(X_train, y_train)

# Get the best parameters and best estimator
best_params = grid_search.best_params_
best_model = grid_search.best_estimator_

print("Best parameters found: ", best_params)

# ---------- Validation evaluation ----------
y_pred_val = best_model.predict(X_val)
Svm_val_accuracy = accuracy_score(y_val, y_pred_val)
print(f"\nValidation accuracy: {Svm_val_accuracy:.4f}")
for avg in ['macro', 'micro', 'weighted']:
    print(f"\nValidation ({avg.upper()}):")
    print(f"Precision: {precision_score(y_val, y_pred_val, average=avg):.4f}")
    print(f"Recall:    {recall_score(y_val, y_pred_val, average=avg):.4f}")
    print(f"F1-score:  {f1_score(y_val, y_pred_val, average=avg):.4f}")

# ---------- Testing evaluation ----------
y_pred_test = best_model.predict(X_test)
Svm_test_accuracy = accuracy_score(y_test, y_pred_test)
print(f"\nTesting accuracy: {Svm_test_accuracy:.4f}")
for avg in ['macro', 'micro', 'weighted']:
    print(f"\nTesting ({avg.upper()}):")
    print(f"Precision: {precision_score(y_test, y_pred_test, average=avg):.4f}")
    print(f"Recall:    {recall_score(y_test, y_pred_test, average=avg):.4f}")
    print(f"F1-score:  {f1_score(y_test, y_pred_test, average=avg):.4f}")



Fitting 5 folds for each of 32 candidates, totalling 160 fits
Best parameters found:  {'C': 0.1, 'gamma': 'scale', 'kernel': 'linear'}

Validation accuracy: 0.8755

Validation (MACRO):
Precision: 0.2918
Recall:    0.3333
F1-score:  0.3112

Validation (MICRO):
Precision: 0.8755
Recall:    0.8755
F1-score:  0.8755

Validation (WEIGHTED):
Precision: 0.7664
Recall:    0.8755
F1-score:  0.8173

Testing accuracy: 0.8778

Testing (MACRO):
Precision: 0.2926
Recall:    0.3333
F1-score:  0.3116

Testing (MICRO):
Precision: 0.8778
Recall:    0.8778
F1-score:  0.8778

Testing (WEIGHTED):
Precision: 0.7705
Recall:    0.8778
F1-score:  0.8206


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/m

In [ ]:
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Define the parameter grid to search
param_grid = {
    'n_estimators': [50, 100, 150],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'criterion': ['gini', 'entropy']
}

# Initialize the Extra Trees classifier
et_model = ExtraTreesClassifier(random_state=42)

# Use GridSearchCV to find the best hyperparameters
grid_search = GridSearchCV(estimator=et_model, param_grid=param_grid, cv=5,
                           scoring='accuracy', verbose=1, n_jobs=-1)

# Fit GridSearchCV to the training data
grid_search.fit(X_train, y_train)

# Get the best parameters and the best estimator
best_params = grid_search.best_params_
best_model = grid_search.best_estimator_

print("Best parameters found: ", best_params)

# ---- Function to print metrics for any split ----
def evaluate_model(name, y_true, y_pred):
    print(f"\n{name} metrics (GridSearchCV):")
    print("Accuracy: {:.2f}%".format(accuracy_score(y_true, y_pred) * 100))

    # Weighted
    print("Precision (weighted): {:.4f}".format(precision_score(y_true, y_pred, average='weighted')))
    print("Recall (weighted): {:.4f}".format(recall_score(y_true, y_pred, average='weighted')))
    print("F1 Score (weighted): {:.4f}".format(f1_score(y_true, y_pred, average='weighted')))

    # Macro
    print("Precision (macro): {:.4f}".format(precision_score(y_true, y_pred, average='macro')))
    print("Recall (macro): {:.4f}".format(recall_score(y_true, y_pred, average='macro')))
    print("F1 Score (macro): {:.4f}".format(f1_score(y_true, y_pred, average='macro')))

    # Micro
    print("Precision (micro): {:.4f}".format(precision_score(y_true, y_pred, average='micro')))
    print("Recall (micro): {:.4f}".format(recall_score(y_true, y_pred, average='micro')))
    print("F1 Score (micro): {:.4f}".format(f1_score(y_true, y_pred, average='micro')))

# Evaluate on validation data
evaluate_model("Validation", y_val, best_model.predict(X_val))

# Evaluate on testing data
evaluate_model("Testing", y_test, best_model.predict(X_test))


Fitting 5 folds for each of 216 candidates, totalling 1080 fits
Best parameters found:  {'criterion': 'gini', 'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 100}

Validation metrics (GridSearchCV):
Accuracy: 87.55%
Precision (weighted): 0.7664
Recall (weighted): 0.8755
F1 Score (weighted): 0.8173
Precision (macro): 0.2918
Recall (macro): 0.3333
F1 Score (macro): 0.3112
Precision (micro): 0.8755
Recall (micro): 0.8755
F1 Score (micro): 0.8755

Testing metrics (GridSearchCV):
Accuracy: 88.33%
Precision (weighted): 0.8720
Recall (weighted): 0.8833
F1 Score (weighted): 0.8336
Precision (macro): 0.6276
Recall (macro): 0.3524
F1 Score (macro): 0.3486
Precision (micro): 0.8833
Recall (micro): 0.8833
F1 Score (micro): 0.8833


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/m

In [ ]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Initialize the AdaBoost classifier
n_estimators = 100
learning_rate = 1.0
algorithm = 'SAMME'
model = AdaBoostClassifier(n_estimators=n_estimators,
                           learning_rate=learning_rate,
                           algorithm=algorithm)

# Train the AdaBoost classifier
model.fit(X_train, y_train)

# ---------- Validation evaluation ----------
y_pred_val = model.predict(X_val)
Ada_val_accuracy = accuracy_score(y_val, y_pred_val)
print(f"\nValidation accuracy: {Ada_val_accuracy:.4f}")
for avg in ['macro', 'micro', 'weighted']:
    print(f"\nValidation ({avg.upper()}):")
    print(f"Precision: {precision_score(y_val, y_pred_val, average=avg):.4f}")
    print(f"Recall:    {recall_score(y_val, y_pred_val, average=avg):.4f}")
    print(f"F1-score:  {f1_score(y_val, y_pred_val, average=avg):.4f}")

# ---------- Testing evaluation ----------
y_pred_test = model.predict(X_test)
Ada_test_accuracy = accuracy_score(y_test, y_pred_test)
print(f"\nTesting accuracy: {Ada_test_accuracy:.4f}")
for avg in ['macro', 'micro', 'weighted']:
    print(f"\nTesting ({avg.upper()}):")
    print(f"Precision: {precision_score(y_test, y_pred_test, average=avg):.4f}")
    print(f"Recall:    {recall_score(y_test, y_pred_test, average=avg):.4f}")
    print(f"F1-score:  {f1_score(y_test, y_pred_test, average=avg):.4f}")


/usr/local/lib/python3.11/dist-packages/sklearn/ensemble/_weight_boosting.py:519: FutureWarning: The parameter 'algorithm' is deprecated in 1.6 and has no effect. It will be removed in version 1.8.
  warnings.warn(



Validation accuracy: 0.8699

Validation (MACRO):
Precision: 0.2916
Recall:    0.3312
F1-score:  0.3101

Validation (MICRO):
Precision: 0.8699
Recall:    0.8699
F1-score:  0.8699

Validation (WEIGHTED):
Precision: 0.7658
Recall:    0.8699
F1-score:  0.8145

Testing accuracy: 0.8778

Testing (MACRO):
Precision: 0.2926
Recall:    0.3333
F1-score:  0.3116

Testing (MICRO):
Precision: 0.8778
Recall:    0.8778
F1-score:  0.8778

Testing (WEIGHTED):
Precision: 0.7705
Recall:    0.8778
F1-score:  0.8206


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/m

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Define the parameter grid to search
param_grid = {
    'n_estimators': [50, 100, 150],
    'max_depth': [10, 20, 30, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'bootstrap': [True, False]
}

# Initialize the Random Forest classifier
rf_model = RandomForestClassifier(random_state=42)

# Use GridSearchCV to find the best hyperparameters
grid_search = GridSearchCV(estimator=rf_model, param_grid=param_grid, cv=5, scoring='accuracy', verbose=1, n_jobs=-1)

# Fit GridSearchCV to the training data
grid_search.fit(X_train, y_train)

# Get the best parameters and best estimator
best_params = grid_search.best_params_
best_model = grid_search.best_estimator_

print("Best parameters found: ", best_params)

# ---------- Validation evaluation ----------
y_pred_val = best_model.predict(X_val)
Rf_val_accuracy = accuracy_score(y_val, y_pred_val)
print(f"\nValidation accuracy: {Rf_val_accuracy:.4f}")
for avg in ['macro', 'micro', 'weighted']:
    print(f"\nValidation ({avg.upper()}):")
    print(f"Precision: {precision_score(y_val, y_pred_val, average=avg):.4f}")
    print(f"Recall:    {recall_score(y_val, y_pred_val, average=avg):.4f}")
    print(f"F1-score:  {f1_score(y_val, y_pred_val, average=avg):.4f}")

# ---------- Testing evaluation ----------
y_pred_test = best_model.predict(X_test)
Rf_test_accuracy = accuracy_score(y_test, y_pred_test)
print(f"\nTesting accuracy: {Rf_test_accuracy:.4f}")
for avg in ['macro', 'micro', 'weighted']:
    print(f"\nTesting ({avg.upper()}):")
    print(f"Precision: {precision_score(y_test, y_pred_test, average=avg):.4f}")
    print(f"Recall:    {recall_score(y_test, y_pred_test, average=avg):.4f}")
    print(f"F1-score:  {f1_score(y_test, y_pred_test, average=avg):.4f}")


In [ ]:
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Initialize the SGD classifier
alpha = 0.0001
max_iter = 1000
tol = 1e-3
model = SGDClassifier(alpha=alpha, max_iter=max_iter, tol=tol)

# Train the SGD classifier
model.fit(X_train, y_train)

# ---------- Validation evaluation ----------
y_pred_val = model.predict(X_val)
Sgd_val_accuracy = accuracy_score(y_val, y_pred_val)
print(f"\nValidation accuracy: {Sgd_val_accuracy:.4f}")
for avg in ['macro', 'micro', 'weighted']:
    print(f"\nValidation ({avg.upper()}):")
    print(f"Precision: {precision_score(y_val, y_pred_val, average=avg):.4f}")
    print(f"Recall:    {recall_score(y_val, y_pred_val, average=avg):.4f}")
    print(f"F1-score:  {f1_score(y_val, y_pred_val, average=avg):.4f}")

# ---------- Testing evaluation ----------
y_pred_test = model.predict(X_test)
Sgd_test_accuracy = accuracy_score(y_test, y_pred_test)
print(f"\nTesting accuracy: {Sgd_test_accuracy:.4f}")
for avg in ['macro', 'micro', 'weighted']:
    print(f"\nTesting ({avg.upper()}):")
    print(f"Precision: {precision_score(y_test, y_pred_test, average=avg):.4f}")
    print(f"Recall:    {recall_score(y_test, y_pred_test, average=avg):.4f}")
    print(f"F1-score:  {f1_score(y_test, y_pred_test, average=avg):.4f}")



Validation accuracy: 0.8643

Validation (MACRO):
Precision: 0.2919
Recall:    0.3291
F1-score:  0.3094

Validation (MICRO):
Precision: 0.8643
Recall:    0.8643
F1-score:  0.8643

Validation (WEIGHTED):
Precision: 0.7666
Recall:    0.8643
F1-score:  0.8126

Testing accuracy: 0.8694

Testing (MACRO):
Precision: 0.4881
Recall:    0.3746
F1-score:  0.3826

Testing (MICRO):
Precision: 0.8694
Recall:    0.8694
F1-score:  0.8694

Testing (WEIGHTED):
Precision: 0.8060
Recall:    0.8694
F1-score:  0.8253


In [ ]:
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, log_loss

# Initialize the base models
lr_model = LogisticRegression(max_iter=1000)
# Change the loss to 'log_loss' for probability estimates
sgd_model = SGDClassifier(loss='log_loss', max_iter=1000, tol=1e-3)

# Create the Voting Classifier (change voting='hard' for hard voting)
vc_model = VotingClassifier(
    estimators=[('lr', lr_model), ('sgd', sgd_model)],
    voting='soft'  # use 'hard' for majority voting
)

# Train the voting classifier
vc_model.fit(X_train, y_train)

# ---------- Validation evaluation ----------
y_pred_val = vc_model.predict(X_val)
vc_val_accuracy = accuracy_score(y_val, y_pred_val)
print(f"\nValidation accuracy: {vc_val_accuracy:.4f}")
for avg in ['macro', 'micro', 'weighted']:
    print(f"\nValidation ({avg.upper()}):")
    print(f"Precision: {precision_score(y_val, y_pred_val, average=avg):.4f}")
    print(f"Recall:    {recall_score(y_val, y_pred_val, average=avg):.4f}")
    print(f"F1-score:  {f1_score(y_val, y_pred_val, average=avg):.4f}")

# ---------- Testing evaluation ----------
y_pred_test = vc_model.predict(X_test)
vc_test_accuracy = accuracy_score(y_test, y_test_pred)
print(f"\nTesting accuracy: {vc_test_accuracy:.4f}")
for avg in ['macro', 'micro', 'weighted']:
    print(f"\nTesting ({avg.upper()}):")
    print(f"Precision: {precision_score(y_test, y_test_pred, average=avg):.4f}")
    print(f"Recall:    {recall_score(y_test, y_test_pred, average=avg):.4f}")
    print(f"F1-score:  {f1_score(y_test, y_test_pred, average=avg):.4f}")